# Diabetes Prediction: Basic Train vs. Test Comparison

Simplest possible version: one dataset, one split, three models trained in separate cells. No grid search, no class weighting, no scaling pipelines.

Goal: compare each model's metrics on the **training set** vs. the **test set**. A big gap (train much better than test) means overfitting. Both scores being low means underfitting.

Run the setup cells once, then run each model's cell independently (in any order).

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier

RANDOM_STATE = 42

## Load data

In [2]:
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

df.shape

(253680, 22)

## Train/test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train.shape, X_test.shape

((202944, 21), (50736, 21))

## Scoring helper

In [4]:
results = {}

def score(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred),
        'recall': recall_score(y, pred),
        'f1': f1_score(y, pred),
    }

def train_and_score(name, model):
    model.fit(X_train, y_train)
    train_scores = score(model, X_train, y_train)
    test_scores = score(model, X_test, y_test)
    results[name] = {'train': train_scores, 'test': test_scores}

    print(f'{name}')
    print(f"  train -> {train_scores}")
    print(f"  test  -> {test_scores}")
    return model

## Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
log_reg = train_and_score('LogisticRegression', log_reg)

LogisticRegression
  train -> {'accuracy': 0.8638392857142857, 'precision': 0.5389547544156786, 'recall': 0.15754853768080065, 'f1': 0.24382234627698876}
  test  -> {'accuracy': 0.8621491643014821, 'precision': 0.5173530772790375, 'recall': 0.15815532607158014, 'f1': 0.24225352112676057}


## Random Forest

In [6]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf = train_and_score('RandomForest', rf)

RandomForest
  train -> {'accuracy': 0.994417179123305, 'precision': 0.9951477562933236, 'recall': 0.9646355695441525, 'f1': 0.9796541383087615}
  test  -> {'accuracy': 0.8594094922737306, 'precision': 0.4876256767208043, 'recall': 0.1783844956853869, 'f1': 0.26121180735370275}


## XGBoost

In [ ]:
xgb = XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE)
xgb = train_and_score('XGBoost', xgb)

XGBoost
  train -> {'accuracy': 0.8765127325764743, 'precision': 0.6809995497523638, 'recall': 0.21395480425787744, 'f1': 0.3256101827184414}
  test  -> {'accuracy': 0.8633711762850835, 'precision': 0.5308976093820478, 'recall': 0.16650162682133257, 'f1': 0.25349989231100584}


## Comparison table

A large positive gap on accuracy/f1 signals overfitting; low scores on both signals underfitting.

In [8]:
rows = []
for name, sets in results.items():
    for set_name, metrics in sets.items():
        rows.append({'model': name, 'set': set_name, **metrics})

summary = pd.DataFrame(rows).round(3)

pivot = summary.pivot(index='model', columns='set', values=['accuracy', 'precision', 'recall', 'f1'])
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    pivot[(metric, 'gap (train-test)')] = (pivot[(metric, 'train')] - pivot[(metric, 'test')]).round(3)
pivot = pivot.sort_index(axis=1, level=0)
pivot

accuracy                             f1         \
set                gap (train-test)   test  train gap (train-test)   test   
model                                                                       
LogisticRegression            0.002  0.862  0.864            0.002  0.242   
RandomForest                  0.135  0.859  0.994            0.719  0.261   
XGBoost                       0.014  0.863  0.877            0.073  0.253   

                                 precision                         recall  \
set                 train gap (train-test)   test  train gap (train-test)   
model                                                                       
LogisticRegression  0.244            0.022  0.517  0.539            0.000   
RandomForest        0.980            0.507  0.488  0.995            0.787   
XGBoost             0.326            0.150  0.531  0.681            0.047   

                                  
set                  test  train  
model                             
LogisticRegression  0.158  0.158  
RandomForest        0.178  0.965  
XGBoost             0.167  0.214

## Tuned models: grid search + class-imbalance handling

The baseline above shows Random Forest badly overfitting (train recall 0.965 vs test 0.178) and both RF and XGBoost ignoring the class imbalance (~86% negative / ~14% positive). Fixes applied below, per notes:

- **Random Forest**: `class_weight='balanced'` (wasn't set before — biggest reason recall was low); cap `max_depth` at `[4, 6, 8]` instead of unbounded (unbounded lets a tree memorize the training set); grid `min_samples_split` as another lever against overfitting; `n_estimators` fixed at 100. Skipping `ccp_alpha` — it's a pruning parameter, and we're already bounding tree size with `max_depth`, so tuning both is redundant.
- **XGBoost**: `scale_pos_weight` set to the actual negative/positive ratio in the training data (wasn't set before); grid `learning_rate` (eta) over `[0.1, 0.2]` and `max_depth` over `[3, 4, 6]`.
- **Logistic Regression**: added a `class_weight='balanced'` variant so all three models get the same imbalance handling — otherwise the comparison isn't apples-to-apples.
- **Validation**: `GridSearchCV` with `StratifiedKFold` does the "hold out and validate" step internally via k-fold, so there's no need for a separate manual validation split — each fold acts as a validation set in turn (this is the k-fold approach from your notes).
- **Selection**: tune on recall (`scoring='recall'`), but the last cell below checks precision too — a model that just predicts "positive" for everyone gets perfect recall and useless precision, so recall alone isn't enough to pick a winner.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### Random Forest (tuned)

In [ ]:
rf_grid = {
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5, 10],
}

rf_gs = GridSearchCV(
    RandomForestClassifier(n_estimators=100, class_weight='balanced',
                           random_state=RANDOM_STATE, n_jobs=-1),
    rf_grid, scoring='recall', cv=cv, n_jobs=-1
)
rf_gs.fit(X_train, y_train)
rf_best = rf_gs.best_estimator_

print('best params:', rf_gs.best_params_)
print('cv recall  :', round(rf_gs.best_score_, 3))

results['RandomForest_tuned'] = {
    'train': score(rf_best, X_train, y_train),
    'test': score(rf_best, X_test, y_test),
}
print(results['RandomForest_tuned'])

### XGBoost (tuned)

In [ ]:
# scale_pos_weight = (#negatives / #positives) in the training data
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print('scale_pos_weight:', round(spw, 2))

xgb_grid = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.1, 0.2],
}

xgb_gs = GridSearchCV(
    XGBClassifier(n_estimators=100, scale_pos_weight=spw, eval_metric='logloss',
                 random_state=RANDOM_STATE, n_jobs=-1),
    xgb_grid, scoring='recall', cv=cv, n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
xgb_best = xgb_gs.best_estimator_

print('best params:', xgb_gs.best_params_)
print('cv recall  :', round(xgb_gs.best_score_, 3))

results['XGBoost_tuned'] = {
    'train': score(xgb_best, X_train, y_train),
    'test': score(xgb_best, X_test, y_test),
}
print(results['XGBoost_tuned'])

### Logistic Regression (class-weighted, for a fair comparison)

In [ ]:
log_reg_balanced = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
log_reg_balanced = train_and_score('LogisticRegression_balanced', log_reg_balanced)

## Updated comparison table (baseline + tuned)

In [ ]:
rows = []
for name, sets in results.items():
    for set_name, metrics in sets.items():
        rows.append({'model': name, 'set': set_name, **metrics})

summary2 = pd.DataFrame(rows).round(3)
pivot2 = summary2.pivot(index='model', columns='set', values=['accuracy', 'precision', 'recall', 'f1'])
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    pivot2[(metric, 'gap (train-test)')] = (pivot2[(metric, 'train')] - pivot2[(metric, 'test')]).round(3)
pivot2 = pivot2.sort_index(axis=1, level=0)
pivot2

### Pick the winner: best test recall, sanity-checked against precision

Recall alone can be gamed — a model that predicts "positive" for every row gets recall=1.0, but its precision is just the positive rate in the data (~14%), which is useless. Compare the winning model's precision against that floor before trusting the recall number.

In [ ]:
test_recall = {name: sets['test']['recall'] for name, sets in results.items()}
best_model_name = max(test_recall, key=test_recall.get)

print('Highest test recall:', best_model_name, '->', round(test_recall[best_model_name], 3))
print('Its test precision :', round(results[best_model_name]['test']['precision'], 3))

baseline_positive_rate = y_test.mean()
print(f"\n'Always predict positive' baseline: recall=1.0, precision={baseline_positive_rate:.3f}")
print('If the winning model\'s precision is close to that baseline, its recall win is not meaningful.')